# PyTorch basics

Keras hides the training loop inside `fit`. PyTorch hands it to you. That is more typing, and it is also where you see what training is: a forward pass, a loss, a backward pass and a weight update, repeated. This notebook covers tensors, autograd, `nn.Module` and a hand-written loop on MNIST. Later notebooks in this section and all of sections 06 and 07 use this style.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from common.datasets import load_mnist
from common.plotting import plot_history

SEED = 42
set_seed(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
device = get_device()
torch.backends.cudnn.deterministic = True  # same seed, same numbers on a GPU
print("torch", torch.__version__, "device:", device)

## 1. Intuition

A tensor is a NumPy array that can live on a GPU and remember how it was made. That second part is autograd. Every operation on a tensor with `requires_grad=True` is recorded, so `loss.backward()` can walk the record in reverse and fill in a gradient for each weight. It is the chain rule from `01_linear_algebra_calculus/04_autodiff.ipynb`, run for you.

A model is a bag of tensors (the parameters) plus a function. `nn.Module` holds both and lets you ask for `model.parameters()`. The training loop is four lines. The rest is bookkeeping.

## 2. Maths and cost function

Take one weight and one data point: $L=(wx-y)^2$. The chain rule gives

$$\frac{\partial L}{\partial w}=2\,(wx-y)\,x$$

With $w=2$, $x=3$, $y=10$ that is $2\times(-4)\times3=-24$. The cost function for MNIST is cross-entropy on 10 output logits. PyTorch's `CrossEntropyLoss` includes the softmax, so the model returns raw logits and you never add a softmax layer yourself. The update is plain gradient descent, $w\leftarrow w-\eta\,\partial L/\partial w$.

## 3. From scratch

The building blocks first. Tensors behave like NumPy arrays: shapes, broadcasting, matrix products. `.numpy()` shares memory with the tensor on the CPU, so editing one edits the other.

In [ ]:
a = torch.arange(6, dtype=torch.float32).reshape(2, 3)
b = torch.tensor([10.0, 20.0, 30.0])
print("shape", a.shape, "dtype", a.dtype, "device", a.device)
print("broadcast a + b:\n", a + b)
print("matmul a @ a.T:\n", a @ a.T)
n = a.numpy()  # shares memory with the tensor on the CPU
n[0, 0] = 99
print("tensor changed through the numpy view:", a[0, 0].item())

Now autograd on the single-weight example. The number it returns matches the hand calculation.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
x_pt, y_pt = torch.tensor(3.0), torch.tensor(10.0)
loss = (w * x_pt - y_pt) ** 2
loss.backward()
print(f"loss {loss.item():.1f}, autograd dL/dw {w.grad.item():.1f}")
print(f"by hand: 2 * (w*x - y) * x = {2 * (2.0 * 3.0 - 10.0) * 3.0:.1f}")

## 4. Library version

`nn.Module` packages the parameters and the forward pass. Two `Linear` layers with a ReLU between them is the same 784-128-10 network as notebook 01. PyTorch counts 101,770 parameters, the same as Keras: $784\times128+128+128\times10+10$.

In [ ]:
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)


def to_tensors(x: np.ndarray, y: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    return torch.from_numpy(x.reshape(len(x), -1)), torch.from_numpy(y)


X_tr, Y_tr = to_tensors(x_tr, y_tr)
X_te, Y_te = to_tensors(x_te, y_te)


class MLP(nn.Module):
    def __init__(self, n_in: int = 784, n_hidden: int = 128, n_out: int = 10) -> None:
        super().__init__()
        self.hidden = nn.Linear(n_in, n_hidden)
        self.out = nn.Linear(n_hidden, n_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out(torch.relu(self.hidden(x)))  # logits, no softmax


set_seed(SEED)
model = MLP()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))
for name, p in model.named_parameters():
    print(f"  {name:>14} {tuple(p.shape)}")

Next the loop. Each pass over a batch does the same four things, in this order: clear the old gradients, forward and loss, backward, step. The `accuracy` helper switches to `eval()` and turns gradient tracking off with `no_grad`, which saves memory and time.

Device handling is one line each way. The model goes to the device once with `.to(device)`. Each batch is moved as it is used, because the whole dataset need not fit on the GPU. This loop runs unchanged on `cpu`, `cuda` or `mps`.

In [ ]:
def accuracy(model: nn.Module, x: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    with torch.no_grad():  # no graph needed for evaluation
        return (model(x.to(device)).argmax(dim=1).cpu() == y).float().mean().item()


def train(
    model: nn.Module, epochs: int = 5, lr: float = 0.1, batch_size: int = 64, seed: int = SEED
) -> dict[str, list[float]]:
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed)
    hist: dict[str, list[float]] = {"loss": [], "val_acc": []}
    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(X_tr), generator=g)
        total = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i : i + batch_size]
            xb, yb = X_tr[idx].to(device), Y_tr[idx].to(device)
            opt.zero_grad()  # 1. clear old gradients
            loss = loss_fn(model(xb), yb)  # 2. forward pass and loss
            loss.backward()  # 3. backward pass fills p.grad
            opt.step()  # 4. update the weights
            total += loss.item() * len(idx)
        hist["loss"].append(total / len(order))
        hist["val_acc"].append(accuracy(model, X_te, Y_te))
    return hist


set_seed(SEED)
hist = train(MLP())
print(
    f"loss {hist['loss'][0]:.3f} -> {hist['loss'][-1]:.3f}, test accuracy {hist['val_acc'][-1]:.3f}"
)
plot_history(hist, ("loss",), title="hand-written loop")
plt.show()

Loss falls from 1.37 to 0.34 in five epochs and the test accuracy is 87%. Notebook 01 reached about 91% with Adam and 10 epochs. This loop uses plain SGD for five, so the gap is expected.

## 5. Tuning

The learning rate is the knob that matters most for SGD. Same loop, three values.

In [ ]:
# Same loop, three learning rates.
for lr in (0.01, 0.1, 0.5):
    set_seed(SEED)
    h = train(MLP(), epochs=5, lr=lr)
    print(f"SGD lr {lr:>4}: final loss {h['loss'][-1]:.3f}, test accuracy {h['val_acc'][-1]:.3f}")

At 0.01 the loss is still 1.29 after five epochs, so the model has barely started (79% accuracy). At 0.5 the training loss is the lowest, 0.14, but the test accuracy is 83% against 87% at 0.1. A lower training loss is not a better model. This is what notebook 03 showed with Keras, and the loop makes the cause visible: the rate is one number in one line.

## 6. What breaks

Three common mistakes, each of which runs without an error or fails with a short message. First, forgetting `zero_grad`. PyTorch adds each new gradient to the old one instead of replacing it.

In [ ]:
set_seed(SEED)
m = MLP().to(device)
opt = torch.optim.SGD(m.parameters(), lr=0.1)
xb, yb = X_tr[:64].to(device), Y_tr[:64].to(device)
for step in range(3):  # no opt.zero_grad()
    nn.functional.cross_entropy(m(xb), yb).backward()
    print(f"step {step}: gradient norm of hidden.weight = {m.hidden.weight.grad.norm():.3f}")
m.zero_grad()
nn.functional.cross_entropy(m(xb), yb).backward()
print(f"after zero_grad, one backward: {m.hidden.weight.grad.norm():.3f}")

The gradient norm grows by the same 0.56 every step (0.56, 1.12, 1.68), so by step three the update is three times too big and in a stale direction. After `zero_grad` it is back to 0.56. The loop in section 4 clears it first for this reason.

Second, the mode. Dropout is active in `train()` and off in `eval()`, so the same input gives different outputs in one and the same in the other. Forgetting `model.eval()` before you evaluate gives noisy, lower scores with no warning. Third, dtypes. A float64 array goes into a float32 model and PyTorch stops with a clear message. Convert with `.float()` or build the array as `float32`.

In [ ]:
drop = nn.Sequential(nn.Linear(784, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 10)).to(device)
xs = X_te[:5].to(device)
with torch.no_grad():
    drop.train()
    print("train mode, two passes equal:", torch.equal(drop(xs), drop(xs)))
    drop.eval()
    print("eval mode, two passes equal: ", torch.equal(drop(xs), drop(xs)))
try:
    m(torch.from_numpy(x_te[:3].reshape(3, -1).astype("float64")).to(device))
except RuntimeError as err:
    print("dtype mismatch:", str(err)[:70])

Tensors carry the data, autograd carries the gradients, and the loop is yours. Next: a layer that shares its weights across the image.